In [11]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline

In [4]:
from transformers import pipeline
sentiment_classifier= pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")


Device set to use cpu


In [20]:
sentiment_classifier(" #@%_**#@!")

[{'label': 'POSITIVE', 'score': 0.9393622279167175}]

In [31]:
import accelerate
print(f"Accelerate version: {accelerate.__version__}")

from pysentimiento import create_analyzer

try:
    analyzer = create_analyzer(task="sentiment", lang="pt")
    result = analyzer.predict("gosto de assistir futebol")
    print("Sucesso! Resultado:", result)
except Exception as e:
    print("Erro:", e)

Accelerate version: 1.10.0
Erro: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`


In [ ]:
import os
os.environ['ACCELERATE_VERSION'] = '0.26.0'  # Ou a versão que você tem instalada

from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="pt")
result = analyzer.predict("gosto de assistir futebol")
print(result)

In [45]:
import os
import sys
import accelerate
from unittest.mock import patch

# Forçar a detecção da versão correta
os.environ['ACCELERATE_VERSION'] = accelerate.__version__

# Monkey patch para bypass na verificação
from transformers.training_args import is_accelerate_available

def patched_is_accelerate_available():
    return True

# Aplicar o patch
import transformers.training_args
transformers.training_args.is_accelerate_available = patched_is_accelerate_available

# Também patchar a função específica que causa o erro
original_setup_devices = None

def patch_setup_devices(self):
    """Patch para bypass da verificação de versão do accelerate"""
    try:
        # Tenta o método original primeiro
        if original_setup_devices:
            return original_setup_devices(self)
    except:
        # Fallback: retorna device CPU
        import torch
        return torch.device('cpu')

# Aplicar o patch
from transformers.training_args import TrainingArguments
original_setup_devices = TrainingArguments._setup_devices
TrainingArguments._setup_devices = patch_setup_devices

# Agora tenta importar
try:
    from pysentimiento import create_analyzer
    analyzer = create_analyzer(task="sentiment", lang="pt")
    result = analyzer.predict("gosto de assistir futebol")
    print("Sucesso!", result)
except Exception as e:
    print("Erro mesmo com patch:", e)

Erro mesmo com patch: name 'AcceleratorConfig' is not defined


In [36]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np

def analyze_sentiment(text):
    # Modelo específico para sentiment analysis em português
    model_name = "pysentimiento/robertuito-sentiment-analysis"
    
    # Carregar modelo e tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    # Tokenizar texto
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    
    # Fazer predição
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
    
    # Mapear resultados
    labels = ["NEG", "NEU", "POS"]
    scores = probabilities[0].tolist()
    
    return {label: score for label, score in zip(labels, scores)}

# Testar
result = analyze_sentiment("eu  gosto de assistir futebol")
print("Resultado:", result)

Resultado: {'NEG': 0.4433320164680481, 'NEU': 0.27209147810935974, 'POS': 0.2845764458179474}


In [44]:
# Alternativa: usar a biblioteca transformers diretamente
from transformers import pipeline

# Criar pipeline de análise de sentimentos em português
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis",
    tokenizer="pysentimiento/robertuito-sentiment-analysis"
)
frase1="hoje estou com muito sono,mas estou legal"
frase2="jogar donkey kong é inclivel"
frase3= "nao gosto de tomar remedio"

result = sentiment_analyzer(frase1)
print("Resultado com pipeline:", result)

Resultado com pipeline: [{'label': 'POS', 'score': 0.4582642614841461}]


In [1]:
import pandas as pd

In [ ]:
df= pd.read_csv("dados/frases.csv")


In [7]:
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis",
    tokenizer="pysentimiento/robertuito-sentiment-analysis"
)
frases = df["frase"].tolist()

result = sentiment_analyzer(frases)
print("Resultado com pipeline:", result)

Device set to use cpu


Resultado com pipeline: [{'label': 'POS', 'score': 0.9259828329086304}, {'label': 'NEG', 'score': 0.918696939945221}, {'label': 'POS', 'score': 0.9112487435340881}, {'label': 'NEG', 'score': 0.9183734059333801}, {'label': 'POS', 'score': 0.8698619604110718}, {'label': 'NEG', 'score': 0.9281649589538574}, {'label': 'POS', 'score': 0.9010726809501648}, {'label': 'NEG', 'score': 0.898954451084137}, {'label': 'POS', 'score': 0.8043502569198608}, {'label': 'NEG', 'score': 0.9088923335075378}, {'label': 'POS', 'score': 0.9325507283210754}, {'label': 'NEG', 'score': 0.903983473777771}, {'label': 'POS', 'score': 0.7416865229606628}, {'label': 'NEG', 'score': 0.8703842163085938}, {'label': 'POS', 'score': 0.9229539036750793}, {'label': 'NEG', 'score': 0.9144430160522461}, {'label': 'POS', 'score': 0.436163067817688}, {'label': 'NEG', 'score': 0.8937331438064575}, {'label': 'POS', 'score': 0.8641861081123352}, {'label': 'NEG', 'score': 0.8896104693412781}, {'label': 'POS', 'score': 0.93692773580

In [ ]:
##Para adicionar os resultados de volta ao DataFrame:

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis",
    tokenizer="pysentimiento/robertuito-sentiment-analysis"
)

frases = df["frase"].tolist()

# Fazer a análise de sentimento
resultados = sentiment_analyzer(frases)

# Adicionar resultados ao DataFrame
df['sentimento'] = [result['label'] for result in resultados]
df['score'] = [result['score'] for result in resultados]

print("DataFrame com resultados:")
print(df.head())

# Estatísticas dos sentimentos
print("\nDistribuição de sentimentos:")
print(df['sentimento'].value_counts())

In [11]:
sentimento = []
for result in resultados:
    sentimento.append(result['label'])  # Para pegar apenas o label (NEG, NEU, POS)

# Ou para pegar ambos label e score
sentimentos_completos = []
for result in resultados:
    sentimentos_completos.append({
        'label': result['label'],
        'score': result['score']
    })

In [ ]:
sentimento

In [14]:
df["Sentimento"]=sentimento
df

,id,frase,sentimento,score,Sentimento
0,1,Adorei o filme que assisti ontem!,POS,0.925983,POS
1,2,"Que serviço péssimo, nunca mais volto aqui.",NEG,0.918697,NEG
2,3,Estou muito feliz com o resultado.,POS,0.911249,POS
3,4,Que dia mais chato e monótono.,NEG,0.918373,NEG
4,5,"Produto de excelente qualidade, recomendo.",POS,0.869862,POS
...,...,...,...,...,...
95,96,Que atendimento personalizado.,POS,0.662149,POS
96,97,"Praia paradisíaca, natureza preservada.",POS,0.763336,POS
97,98,Que filme realista.,POS,0.468528,POS
98,99,Estou honrado pela indicação.,POS,0.888504,POS
